In [2]:
import re

# 1. Definimos um texto de teste
#texto_bruto = "Olá, mundo! Este é o nosso primeiro teste de tokenização."
texto_bruto = "token1 token2 token3 teste"
# 2. Usamos uma expressão regular (regex) para separar palavras e pontuações
# Essa regra diz: "corte o texto sempre que encontrar um espaço OU uma pontuação"
resultado_bruto = re.split(r'([,.:;?_!"()\']|--|\s)', texto_bruto)

# 3. Limpamos a lista removendo espaços vazios indesejados
tokens = [item for item in resultado_bruto if item.strip()]

print("--- RESULTADO DA TOKENIZAÇÃO ---")
print(f"Texto original: {texto_bruto}")
print(f"Quantidade de tokens: {len(tokens)}")
print(f"Lista de tokens: {tokens}")

--- RESULTADO DA TOKENIZAÇÃO ---
Texto original: token1 token2 token3 teste
Quantidade de tokens: 4
Lista de tokens: ['token1', 'token2', 'token3', 'teste']


In [3]:
# --- PASSO 2: Vocabulário e Tokens Especiais ---

# 1. Pegamos os tokens únicos e adicionamos os tokens especiais do GPT
tokens_unicos = sorted(set(tokens))
tokens_unicos.extend(["<|unk|>", "<|endoftext|>"])

# 2. Criamos o dicionário do vocabulário (Token -> ID)
vocab = {token: id_num for id_num, token in enumerate(tokens_unicos)}

# 3. Função para converter tokens em IDs lidando com palavras desconhecidas
def tokens_para_ids(lista_tokens, dicionario_vocab):
    ids = []
    for t in lista_tokens:
        # Se a palavra existir no vocabulário, pega o ID dela; senão, usa o ID de <|unk|>
        id_encontrado = dicionario_vocab.get(t, dicionario_vocab["<|unk|>"])
        ids.append(id_encontrado)
    return ids

# 4. Convertendo a frase original
token_ids = tokens_para_ids(tokens, vocab)

# 5. Testando com uma palavra nova ("carro") que NÃO está no texto inicial
frase_teste = ["token1", "carro", "teste"]
ids_teste = tokens_para_ids(frase_teste, vocab)

print(f"Tamanho do Vocabulário: {len(vocab)}")
print(f"Vocabulário: {vocab}\n")
print(f"Frase original em IDs: {token_ids}")
print(f"Teste com 'carro' (palavra desconhecida): {ids_teste}")

Tamanho do Vocabulário: 6
Vocabulário: {'teste': 0, 'token1': 1, 'token2': 2, 'token3': 3, '<|unk|>': 4, '<|endoftext|>': 5}

Frase original em IDs: [1, 2, 3, 0]
Teste com 'carro' (palavra desconhecida): [1, 4, 0]


In [5]:
# --- PASSO 3: Preparação das Sequências (Entradas e Alvos) ---

# Texto mais longo para podermos criar várias sequências
texto_treino = "isso é um segundo teste de controle e segurança do funcionamento de tokens"
tokens_treino = [item for item in re.split(r'([,.:;?_!"()\']|--|\s)', texto_treino) if item.strip()]

# Atualizamos nosso vocabulário para esse texto
vocab_treino = {token: id_num for id_num, token in enumerate(sorted(set(tokens_treino)))}
vocab_inverso = {id_num: token for token, id_num in vocab_treino.items()}
ids_treino = [vocab_treino[t] for t in tokens_treino]

# Tamanho do contexto: quantos tokens o modelo olha para tentar adivinhar o próximo
tamanho_contexto = 4

entradas = [] # x
alvos = []    # y

# Janela deslizante que percorre o texto criando os pares
for i in range(len(ids_treino) - tamanho_contexto):
    x = ids_treino[i : i + tamanho_contexto]
    y = ids_treino[i + 1 : i + tamanho_contexto + 1]
    entradas.append(x)
    alvos.append(y)

# Vamos inspecionar as duas primeiras amostras criadas
print(f"Total de pares (amostras) gerados: {len(entradas)}\n")

for i in range(2):
    txt_x = [vocab_inverso[idx] for idx in entradas[i]]
    txt_y = [vocab_inverso[idx] for idx in alvos[i]]
    print(f"Amostra {i+1}:")
    print(f"  Entrada (x) [IDs: {entradas[i]}]: {txt_x}")
    print(f"  Alvo    (y) [IDs: {alvos[i]}]: {txt_y}\n")

Total de pares (amostras) gerados: 9

Amostra 1:
  Entrada (x) [IDs: [5, 11, 10, 6]]: ['isso', 'é', 'um', 'segundo']
  Alvo    (y) [IDs: [11, 10, 6, 8]]: ['é', 'um', 'segundo', 'teste']

Amostra 2:
  Entrada (x) [IDs: [11, 10, 6, 8]]: ['é', 'um', 'segundo', 'teste']
  Alvo    (y) [IDs: [10, 6, 8, 1]]: ['um', 'segundo', 'teste', 'de']



In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

# --- PASSO 4: Criando o Dataset e o DataLoader ---

class GPTDataset(Dataset):
    def __init__(self, x_lista, y_lista):
        # Convertemos as listas comuns de Python em Tensores do PyTorch
        self.x = torch.tensor(x_lista)
        self.y = torch.tensor(y_lista)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# 1. Instanciamos o Dataset com as entradas e alvos criados anteriormente
dataset = GPTDataset(entradas, alvos)

# 2. Criamos o DataLoader definindo o tamanho do lote (batch_size = 2)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# 3. Retiramos o primeiro lote para inspecionar
lote_x, lote_y = next(iter(dataloader))

print(f"Formato (shape) do lote de Entrada (x): {lote_x.shape}")
print(f"Formato (shape) do lote de Alvo (y):    {lote_y.shape}\n")
print(f"Conteúdo do Lote de Entrada (x):\n{lote_x}\n")
print(f"Conteúdo do Lote de Alvo (y):\n{lote_y}")

Formato (shape) do lote de Entrada (x): torch.Size([2, 4])
Formato (shape) do lote de Alvo (y):    torch.Size([2, 4])

Conteúdo do Lote de Entrada (x):
tensor([[3, 7, 2, 4],
        [7, 2, 4, 1]])

Conteúdo do Lote de Alvo (y):
tensor([[7, 2, 4, 1],
        [2, 4, 1, 9]])


In [7]:
import torch.nn as nn

# --- PASSO 5: Embeddings e Positional Embeddings ---

tamanho_vocab = len(vocab_treino)
dimensao_emb = 16  # Dimensão do vetor (d_out). No GPT-3 real são 12288, aqui usamos 16 para testar
tamanho_contexto = 4

# 1. Camada de Embedding de Tokens (Palavras)
camada_emb_token = nn.Embedding(tamanho_vocab, dimensao_emb)

# 2. Camada de Embedding de Posição
camada_emb_posicao = nn.Embedding(tamanho_contexto, dimensao_emb)

# 3. Aplicando os Embeddings no nosso Lote de dados (lote_x)
emb_tokens = camada_emb_token(lote_x)

# Criamos a sequência de posições [0, 1, 2, 3] para cada amostra do lote
posicoes = torch.arange(tamanho_contexto)
emb_posicoes = camada_emb_posicao(posicoes)

# 4. Soma Vetorial (Token Embedding + Positional Embedding)
entrada_final = emb_tokens + emb_posicoes

print(f"1. Formato original (Token IDs):          {lote_x.shape}")
print(f"2. Formato dos Embeddings de Tokens:        {emb_tokens.shape}")
print(f"3. Formato dos Embeddings de Posição:       {emb_posicoes.shape}")
print(f"4. Formato FINAL (Entrada da Rede Neural):  {entrada_final.shape}")

1. Formato original (Token IDs):          torch.Size([2, 4])
2. Formato dos Embeddings de Tokens:        torch.Size([2, 4, 16])
3. Formato dos Embeddings de Posição:       torch.Size([4, 16])
4. Formato FINAL (Entrada da Rede Neural):  torch.Size([2, 4, 16])


In [8]:
# --- PASSO 6: Experimentos com Parâmetros ---

def rodar_experimento(tamanho_ctx, tamanho_lote, dim_emb):
    # Reconstruindo dataset com o novo contexto
    entradas_exp, alvos_exp = [], []
    for i in range(len(ids_treino) - tamanho_ctx):
        entradas_exp.append(ids_treino[i : i + tamanho_ctx])
        alvos_exp.append(ids_treino[i + 1 : i + tamanho_ctx + 1])
    
    loader = DataLoader(GPTDataset(entradas_exp, alvos_exp), batch_size=tamanho_lote, shuffle=False)
    x_batch, _ = next(iter(loader))
    
    emb_tok = nn.Embedding(len(vocab_treino), dim_emb)
    emb_pos = nn.Embedding(tamanho_ctx, dim_emb)
    out = emb_tok(x_batch) + emb_pos(torch.arange(tamanho_ctx))
    
    print(f"Configuração: Contexto={tamanho_ctx} | Batch={tamanho_lote} | Dim={dim_emb}")
    print(f"  -> Total de amostras geradas: {len(entradas_exp)}")
    print(f"  -> Formato do Tensor Final:   {out.shape}\n")

print("--- RESULTADOS DOS EXPERIMENTOS ---")
rodar_experimento(tamanho_ctx=2, tamanho_lote=2, dim_emb=16)
rodar_experimento(tamanho_ctx=4, tamanho_lote=4, dim_emb=16)
rodar_experimento(tamanho_ctx=4, tamanho_lote=2, dim_emb=64)

--- RESULTADOS DOS EXPERIMENTOS ---
Configuração: Contexto=2 | Batch=2 | Dim=16
  -> Total de amostras geradas: 11
  -> Formato do Tensor Final:   torch.Size([2, 2, 16])

Configuração: Contexto=4 | Batch=4 | Dim=16
  -> Total de amostras geradas: 9
  -> Formato do Tensor Final:   torch.Size([4, 4, 16])

Configuração: Contexto=4 | Batch=2 | Dim=64
  -> Total de amostras geradas: 9
  -> Formato do Tensor Final:   torch.Size([2, 4, 64])

